<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/treaper33-hue/ML-AI-assisted-/blob/main/compilers_and_triton/notebooks/b2_softmax.ipynb)

# B2 — Reductions, and why softmax is written oddly

**INTERMEDIATE** · runtime a few seconds, mostly compilation

---

A reduction is the first thing that breaks the elementwise picture:
softmax needs the maximum and the sum of a row before it can emit
anything. Two lessons — give each program instance a whole row, and
**subtract the maximum**, because `exp(x)` overflows float32 above about
88 and logits of 100 are ordinary.

> **This stage needs a GPU to verify its kernel.** In Colab: **Runtime ->
> Change runtime type -> T4 GPU**, then re-run the setup cell. Triton ships
> with Colab's PyTorch, so nothing needs installing.
>
> Without a GPU the module still runs and still teaches: the NumPy reference
> is checked and the kernel is reported as `SKIP`. **A skip is not a pass.**

---

## How to use this notebook

1. Run **Setup**. It defines the stage and runs nothing.
2. Run the stage and read its output, gates included.
3. Work through the sections below, which take the pieces apart and
   invite you to break them.

The gates are declared *before* the run. When one fails the useful question
is which is wrong — the code, or the threshold someone wrote down before the
problem was understood.

## Setup

Defines everything. Runs nothing.

In [ ]:
#@title Setup — run this cell first (collapsed: it is the whole stage)
# ===== embedded: track_b_triton/runtime.py =====
"""Shared runtime for the Triton track.

Triton compiles and runs on NVIDIA and AMD GPUs. Most people learning
it do not have one to hand, and official Triton wheels are Linux-only,
so every module in this track is built the same way:

1. a **reference implementation** in NumPy that states, unambiguously,
   what the kernel is supposed to compute;
2. a **Triton kernel** that computes it fast;
3. a **check** that runs the reference always and the kernel when a GPU
   is present.

Without a GPU the modules still run, still teach, and report the kernel
as `SKIPPED` rather than pretending. A skip is not a pass — the same
rule as the rest of this repository.

The fastest way to get a GPU for this track is Colab: Runtime → Change
runtime type → T4 GPU. Triton ships with PyTorch there, so nothing
needs installing.
"""

import dataclasses
import time
from typing import Callable, List, Optional

import numpy as np

try:
    import torch
    HAS_TORCH = True
except ImportError:                         # pragma: no cover
    HAS_TORCH = False

try:
    import triton
    import triton.language as tl
    HAS_TRITON = True
except ImportError:                         # pragma: no cover
    HAS_TRITON = False
    triton = None
    tl = None

HAS_GPU = bool(HAS_TORCH and torch.cuda.is_available())
CAN_RUN_KERNELS = bool(HAS_TRITON and HAS_GPU)

PASS = 'PASS'
FAIL = 'FAIL'
SKIP = 'SKIP'


def environment_banner() -> str:
    """Describes what this machine can actually verify.

    Returns:
        A multi-line description.
    """
    lines = ['environment:']
    if HAS_TORCH:
        lines.append(f'  torch    {torch.__version__}')
        if HAS_GPU:
            major, minor = torch.cuda.get_device_capability(0)
            lines.append(f'  gpu      '
                         f'{torch.cuda.get_device_name(0)} '
                         f'(sm_{major}{minor})')
        else:
            lines.append('  gpu      none visible')
    else:
        lines.append('  torch    not installed')
    lines.append(f'  triton   {"installed" if HAS_TRITON else "not installed"}')
    if CAN_RUN_KERNELS:
        lines.append('  -> kernels will RUN and be checked against the '
                     'reference')
    else:
        lines.append('  -> kernels will be SKIPPED; the reference still '
                     'runs and is checked')
        lines.append('     (Colab: Runtime -> Change runtime type -> T4 GPU)')
    return '\n'.join(lines)


@dataclasses.dataclass
class Result:
    """The outcome of one checked kernel.

    Attributes:
        name: What was checked.
        status: PASS, FAIL or SKIP.
        detail: Numbers a reader can judge.
        milliseconds: Kernel time, when it ran.
    """

    name: str
    status: str
    detail: str = ''
    milliseconds: Optional[float] = None

    def line(self) -> str:
        """Renders one report line."""
        timing = (f'  {self.milliseconds:7.3f} ms'
                  if self.milliseconds is not None else '')
        return f'  [{self.status}] {self.name:<40}{timing}  {self.detail}'


def relative_error(actual: np.ndarray, expected: np.ndarray) -> float:
    """Returns the largest relative elementwise difference.

    Absolute difference is the wrong measure for anything in float32:
    an error of 1e-3 is catastrophic near zero and invisible near 1e6.

    Args:
        actual: What the kernel produced.
        expected: What the reference produced.

    Returns:
        The maximum relative error.
    """
    actual = np.asarray(actual, dtype=np.float64)
    expected = np.asarray(expected, dtype=np.float64)
    scale = np.maximum(np.abs(expected), 1e-6)
    return float(np.max(np.abs(actual - expected) / scale))


def normwise_error(actual: np.ndarray, expected: np.ndarray) -> float:
    """Returns ||actual - expected|| / ||expected||, Frobenius norm.

    `relative_error` above divides elementwise, which is the right
    measure when every expected value is bounded away from zero -- a
    softmax probability, a positive activation. It is the wrong measure
    the moment the output straddles zero, because it is then APE, and
    APE is unbounded at zeros: an absolute error of 1e-4 against an
    entry that happens to be 1e-6 reads as 1e+02 and says nothing about
    the computation.

    A matrix product of random data straddles zero everywhere. This is
    not a hypothetical: the elementwise measure reports 6.2e-02 for
    `torch.matmul` on one of B3's own test shapes, so a gate built on
    it rejects the reference implementation of the operation. Numerical
    linear algebra uses the normwise measure for exactly this reason,
    and so does B3.

    Args:
        actual: What the kernel produced.
        expected: What the reference produced.

    Returns:
        The normwise relative error.
    """
    actual = np.asarray(actual, dtype=np.float64)
    expected = np.asarray(expected, dtype=np.float64)
    scale = float(np.linalg.norm(expected.reshape(-1)))
    return float(np.linalg.norm((actual - expected).reshape(-1))
                 / max(scale, 1e-300))


def check(name: str, kernel: Callable[[], 'np.ndarray'],
          expected: np.ndarray, tolerance: float = 2e-2,
          repeats: int = 20, metric: str = 'elementwise') -> Result:
    """Runs a kernel, times it, and compares it with the reference.

    Args:
        name: What is being checked.
        kernel: Zero-argument callable returning the kernel's output.
        expected: The reference result.
        tolerance: Maximum acceptable relative error. The default is
            loose because float32 accumulation over a reduction is
            genuinely imprecise; each module tightens it where it can.
        repeats: Timed iterations after a warm-up.
        metric: 'elementwise' (the default) or 'normwise'. Choosing
            this per operation rather than once for the whole suite is
            deliberate -- see `normwise_error` for which is right when.

    Returns:
        The result record.

    Raises:
        ValueError: On an unknown metric name.
    """
    if metric not in ('elementwise', 'normwise'):
        raise ValueError(f'unknown metric {metric!r}')
    if not CAN_RUN_KERNELS:
        return Result(name, SKIP, 'no GPU or Triton in this runtime')
    try:
        output = kernel()                   # warm-up: compiles the kernel
        torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(repeats):
            output = kernel()
        torch.cuda.synchronize()
        elapsed = (time.perf_counter() - start) * 1000.0 / repeats

        array = (output.detach().float().cpu().numpy()
                 if HAS_TORCH and isinstance(output, torch.Tensor)
                 else np.asarray(output))
        if metric == 'normwise':
            error = normwise_error(array, expected)
            description = f'normwise relative error {error:.2e}'
        else:
            error = relative_error(array, expected)
            description = f'max relative error {error:.2e}'
        status = PASS if error < tolerance else FAIL
        return Result(name, status, description, elapsed)
    except Exception as error:              # noqa: BLE001 - reported
        return Result(name, FAIL, f'{type(error).__name__}: {error}')


def report(title: str, results: List[Result]) -> int:
    """Prints a result block and returns a process exit code.

    Args:
        title: Section heading.
        results: What to report.

    Returns:
        0 unless something failed. A skip is not a failure, but it is
        also not a pass, and the summary says so.
    """
    print(f'\n{title}')
    for result in results:
        print(result.line())
    failed = sum(r.status == FAIL for r in results)
    skipped = sum(r.status == SKIP for r in results)
    passed = sum(r.status == PASS for r in results)
    print(f'\n  passed {passed}   failed {failed}   skipped {skipped}')
    if skipped and not failed:
        print('  status: UNVERIFIED — the kernels did not run here')
    elif failed:
        print('  status: FAILED')
    else:
        print('  status: VERIFIED')
    return 1 if failed else 0


# ===== track_b_triton/b2_softmax.py =====
"""B2 · INTERMEDIATE — Reductions, and why softmax is written oddly.

Vector add was *elementwise*: every output depends on one input, so
blocks never talk to each other. A reduction is the first thing that
breaks that. Softmax over a row needs the maximum of the row and the
sum of the row before it can produce any output at all.

Two lessons, and the second is the one that bites people.

**1. Give each program instance a whole row.**
If a row fits in a block, the entire reduction happens in registers and
no cross-block communication is needed. One program per row, a block
wide enough for the row, and `tl.max` / `tl.sum` do the rest. When rows
do not fit you need a two-pass algorithm or an online one; this module
takes the common case and says so.

**2. Subtract the maximum, or overflow.**
exp(x) overflows float32 above about 88. Logits of 100 are ordinary.
The identity

    softmax(x) = softmax(x - c)   for any constant c

holds exactly, because the constant cancels between numerator and
denominator. Taking c = max(x) makes the largest exponent exp(0) = 1
and everything else smaller, so nothing overflows. That is why every
softmax implementation you will ever read subtracts the max first, and
it is not a numerical nicety — the naive version returns NaN on real
data.

Run it:
    python b2_softmax.py
"""

import sys
from typing import List

import numpy as np



if CAN_RUN_KERNELS:
    import torch
    import triton
    import triton.language as tl

    @triton.jit
    def softmax_kernel(out_ptr, in_ptr, in_row_stride, out_row_stride,
                       n_cols, BLOCK_SIZE: tl.constexpr):
        """Computes a numerically stable softmax, one row per program.

        Args:
            out_ptr: Output pointer.
            in_ptr: Input pointer.
            in_row_stride: Elements between consecutive input rows.
            out_row_stride: Elements between consecutive output rows.
            n_cols: Row length.
            BLOCK_SIZE: Padded row width; a power of two >= n_cols.
        """
        row = tl.program_id(axis=0)
        columns = tl.arange(0, BLOCK_SIZE)
        mask = columns < n_cols

        # Masked lanes load -inf so they lose the max and contribute
        # exp(-inf) = 0 to the sum. The mask does the padding for free.
        values = tl.load(in_ptr + row * in_row_stride + columns,
                         mask=mask, other=-float('inf'))

        shifted = values - tl.max(values, axis=0)
        numerator = tl.exp(shifted)
        denominator = tl.sum(numerator, axis=0)

        tl.store(out_ptr + row * out_row_stride + columns,
                 numerator / denominator, mask=mask)

    def softmax(x: 'torch.Tensor') -> 'torch.Tensor':
        """Launches one program instance per row.

        Args:
            x: A 2-D tensor on the GPU.

        Returns:
            Row-wise softmax.

        Raises:
            ValueError: If a row is too wide for one block.
        """
        n_rows, n_cols = x.shape
        block_size = triton.next_power_of_2(n_cols)
        if block_size > 65536:
            raise ValueError(
                f'rows of {n_cols} do not fit one block; this kernel '
                f'assumes they do, and a wider row needs the two-pass '
                f'or online algorithm')
        output = torch.empty_like(x)
        softmax_kernel[(n_rows,)](
            output, x, x.stride(0), output.stride(0), n_cols,
            BLOCK_SIZE=block_size,
            num_warps=min(max(block_size // 256, 1), 16))
        return output


def reference_softmax(x: np.ndarray) -> np.ndarray:
    """The specification: stable row-wise softmax.

    Args:
        x: A 2-D array.

    Returns:
        Row-wise softmax, computed in float64.
    """
    x = np.asarray(x, dtype=np.float64)
    shifted = x - x.max(axis=1, keepdims=True)
    numerator = np.exp(shifted)
    return numerator / numerator.sum(axis=1, keepdims=True)


def naive_softmax(x: np.ndarray) -> np.ndarray:
    """The version everybody writes first, in float32, without the shift.

    Kept so the failure can be demonstrated rather than asserted.

    Args:
        x: A 2-D array.

    Returns:
        Row-wise softmax, or NaN where exp overflowed.
    """
    x = np.asarray(x, dtype=np.float32)
    with np.errstate(over='ignore', invalid='ignore'):
        numerator = np.exp(x)
        return numerator / numerator.sum(axis=1, keepdims=True)


def main() -> int:
    """Demonstrates the numerics lesson and checks the kernel.

    Returns:
        A process exit code.
    """
    print('=' * 70)
    print('B2 · INTERMEDIATE — Reductions, and stable softmax')
    print('=' * 70)
    print()
    print(environment_banner())

    print('\n' + '-' * 70)
    print('Why the maximum is subtracted')
    print('-' * 70)
    logits = np.array([[100.0, 101.0, 102.0]], dtype=np.float32)
    naive = naive_softmax(logits)
    stable = reference_softmax(logits)
    print(f'\n  logits               {logits[0]}')
    print(f'  naive   exp(x)/sum   {naive[0]}')
    print(f'  stable  exp(x-max)   {np.round(stable[0], 6)}')
    print('\n  exp(102) = inf in float32, so the naive version divides')
    print('  inf by inf and returns NaN. These are not exotic logits.')

    failures = 0
    naive_broke = bool(np.any(~np.isfinite(naive)))
    stable_fine = bool(np.all(np.isfinite(stable)))
    failures += not (naive_broke and stable_fine)
    print(f'\n  [{"PASS" if naive_broke else "FAIL"}] the naive version '
          f'does produce NaN here')
    print(f'  [{"PASS" if stable_fine else "FAIL"}] the stable version '
          f'does not')

    rows = reference_softmax(np.array([[100.0, 101.0, 102.0],
                                       [-50.0, 0.0, 50.0]]))
    sums_ok = np.allclose(rows.sum(axis=1), 1.0, atol=1e-12)
    failures += not sums_ok
    print(f'  [{"PASS" if sums_ok else "FAIL"}] every row sums to 1 '
          f'(max row error {abs(rows.sum(axis=1) - 1).max():.1e})')

    shift = reference_softmax(np.array([[1.0, 2.0, 3.0]]))
    shifted = reference_softmax(np.array([[1.0, 2.0, 3.0]]) + 1000.0)
    invariant = relative_error(shifted, shift) < 1e-12
    failures += not invariant
    print(f'  [{"PASS" if invariant else "FAIL"}] softmax(x) == '
          f'softmax(x + 1000), as the identity requires')

    rng = np.random.default_rng(20260101)
    results: List[Result] = []
    for n_rows, n_cols in ((128, 256), (1024, 781), (64, 2048)):
        x = (rng.standard_normal((n_rows, n_cols)) * 10).astype(np.float32)
        expected = reference_softmax(x)
        label = f'softmax {n_rows}x{n_cols}'
        if CAN_RUN_KERNELS:
            x_gpu = torch.from_numpy(x).cuda()
            results.append(check(label,
                                 lambda a=x_gpu: softmax(a),
                                 expected,
                                 tolerance=1e-5))
        else:
            results.append(Result(label, 'SKIP',
                                  'no GPU or Triton in this runtime'))

    code = report('Kernel checks', results)

    print('\nWhat to take from this module')
    print('  * A reduction forces a decision the elementwise case did')
    print('    not: who owns the data being reduced. One row per program')
    print('    keeps the whole reduction in registers.')
    print('  * Masked lanes load -inf, not 0. Loading 0 would make an')
    print('    all-negative row return the wrong maximum.')
    print('  * num_warps is tuned to the block width. Too few and the')
    print('    reduction serialises; too many and most sit idle.')
    print('  * The stability trick is exact, not approximate: the')
    print('    constant cancels algebraically.')
    return code if not failures else 1

## Run it

The stage from start to finish, with its gates at the end.

In [ ]:
main()

## Watch float32 run out of room

The identity `softmax(x) = softmax(x - c)` is exact: the constant cancels
between numerator and denominator. What is not exact is `exp` in
float32, which overflows a little above 88. Find the point where the
naive version stops being arithmetic and starts being NaN.

In [ ]:
import numpy as np
for scale in (1, 10, 50, 80, 88, 90, 100, 1000):
    row = np.array([[scale - 1.0, scale, scale + 1.0]])
    naive = naive_softmax(row)[0]
    stable = reference_softmax(row)[0]
    state = 'ok' if np.all(np.isfinite(naive)) else 'NaN'
    print(f'logits around {scale:<5} naive {state:<4} '
          f'stable {np.round(stable, 4)}')

## Why masked lanes load minus infinity

A padded lane must lose the maximum and contribute nothing to the sum.
Loading `0.0` would do neither: on an all-negative row, zero *is* the
maximum, and the answer comes out wrong with no error anywhere.

In [ ]:
import numpy as np
row = np.array([[-5.0, -3.0, -8.0]])
padded_with_zero = np.array([[-5.0, -3.0, -8.0, 0.0]])
print('true softmax of the row       ',
      np.round(reference_softmax(row)[0], 4))
print('with a zero-padded lane       ',
      np.round(reference_softmax(padded_with_zero)[0][:3],
               4))
print('\nthe padding changed the answer, which is why the '
      'kernel loads -inf')

## One row per program, and where that stops working

The whole reduction happens in registers when a row fits a block, so no
instance ever has to talk to another. The launcher refuses rows that do
not fit rather than silently truncating — the point at which you need a
two-pass or online algorithm is a real boundary, and it should be visible.

In [ ]:
import numpy as np
if CAN_RUN_KERNELS:
    import torch
    for cols in (128, 781, 2048, 65536, 131072):
        try:
            got = softmax(torch.randn(4, cols,
                                      device='cuda'))
            print(f'{cols:>7} columns  ok')
        except ValueError as error:
            print(f'{cols:>7} columns  refused: {error}')
else:
    print('needs a GPU; the reference handles any width')
    wide = np.random.default_rng(0).standard_normal(
        (2, 131072))
    print('reference row sums:',
          reference_softmax(wide).sum(axis=1))

## Try it yourself

Compare against PyTorch on a GPU. The interesting number is not the
speedup — it is that a thirty-line kernel matches a library that has had
years of work, because softmax is memory-bound and there is only so much
to win.

In [ ]:
if CAN_RUN_KERNELS:
    import torch
    x = torch.randn(4096, 1024, device='cuda')
    mine = softmax(x)
    theirs = torch.softmax(x, axis=1)
    print('max difference:',
          float((mine - theirs).abs().max()))
else:
    print('needs a GPU: Runtime -> Change runtime type -> T4 GPU')

---

## Where this sits

The script version is `track_b_triton/b2_softmax.py`, and
`compilers_and_triton/run_all.py` runs the whole ladder in order.
`compilers_and_triton/README.md` explains why the compiler track and the
Triton track are one course rather than two.